In [ ]:
# =============================================================================
# LATENT DIFFUSION & STABLE DIFFUSION — diffusion, but in a smaller room
# =============================================================================
#
# Last notebook (DDPM on MNIST): we added static to every PIXEL of a 32×32
# photo and trained a U-Net to peel it off. That works — but at 512×512 RGB
# that's 786,432 numbers per image. The U-Net has to stare at all of them,
# for ~50–1000 reverse steps. Slow. Expensive. Hard to train from scratch.
#
# Stable Diffusion's big idea (Latent Diffusion, Rombach et al.):
#   Don't denoise pixels. Denoise a COMPRESSED sketch of the image instead.
#
# Think of it like this:
#   Pixel DDPM  = sculpting a full-size marble statue, grain by grain.
#   Latent DDPM = sculpting a small clay maquette, then blowing it up to
#                 full size with a fixed "enlarger" (the VAE decoder).
#
#
# ---------------------------------------------------------------------------
# 1) MOTIVATION — from pixel diffusion to latent diffusion
# ---------------------------------------------------------------------------
#
# Pixel space (what notebook 18 did):
#
#   real photo x  ──add noise──▶  snowy photo  ──U-Net peels──▶  new photo
#   shape: [3, 512, 512]  every step, every channel, every pixel
#
# Problems:
#   • Memory: a batch of 512×512 images fills GPU RAM fast.
#   • Compute: U-Net attention over huge feature maps is brutal.
#   • Redundancy: neighboring pixels are highly correlated — we're learning
#     noise patterns the VAE could summarize in fewer numbers.
#
# Latent space (Stable Diffusion):
#
#   real photo x
#       │
#       ▼  VAE encoder (fixed, pretrained)
#   compact latent z   ← e.g. [4, 64, 64] instead of [3, 512, 512]
#       │
#       ▼  add noise + denoise here (the part we train / use)
#   clean latent z₀
#       │
#       ▼  VAE decoder (fixed, pretrained)
#   final image
#
# Same DDPM math as before — just on z instead of x. Roughly 8× smaller
# on each side → ~64× fewer spatial locations. That's why SD fits on
# a consumer GPU.
#
#
# ---------------------------------------------------------------------------
# 2) COMPONENTS OF STABLE DIFFUSION — four main pieces + a scheduler
# ---------------------------------------------------------------------------
#
#   ┌─────────────┐     ┌──────────────────┐     ┌─────────────┐
#   │   PROMPT    │────▶│  CLIP Text Enc.  │────▶│ text embeds │
#   │ "a cat..."  │     │  (frozen)          │     │  for U-Net  │
#   └─────────────┘     └──────────────────┘     └──────┬──────┘
#                                                        │
#   ┌─────────────┐     ┌──────────────────┐              │
#   │ random z_T  │────▶│  U-Net denoiser  │◀─────────────┘
#   │ (latent     │     │  + cross-attn    │   "draw a cat"
#   │  snow)      │◀───▶│  to text         │
#   └─────────────┘     └──────────────────┘
#              ▲                  │
#              │         ┌────────┴────────┐
#              │         │    Scheduler    │  (DDIM, Euler, …)
#              │         │  how to peel    │  — not learned; a recipe
#              │         └─────────────────┘
#              │
#   clean z₀ ──┘
#       │
#       ▼
#   ┌─────────────┐
#   │ VAE Decoder │  ──▶  512×512 RGB image
#   └─────────────┘
#
# Piece by piece:
#
# (A) VAE — the zip / unzip machine (usually FROZEN at inference)
#     Encoder: photo → latent z  (lossy but good enough for art)
#     Decoder: latent z → photo
#     Trained separately on millions of images (reconstruction + KL).
#     You almost never retrain this in a course demo.
#
# (B) CLIP Text Encoder — turns words into vectors the U-Net can read
#     "golden retriever on a beach" → sequence of embeddings.
#     Also frozen. The U-Net learns to *listen* via cross-attention.
#
# (C) U-Net in latent space — the same job as notebook 18, but:
#     • input/output: 4-channel latent, not 3-channel RGB
#     • extra cross-attention layers: "what did the prompt say?"
#     • still gets a timestep t embedding ("how snowy is it?")
#
# (D) Scheduler — the peel recipe
#     Not a neural net. Given predicted noise ε̂, it computes z_{t-1}.
#     DDPM = many small careful steps. DDIM / Euler = fewer, faster steps.
#     Same trained U-Net; different way to walk from z_T → z_0.
#
# (E) Tokenizer — splits prompt into subwords for CLIP (like BPE in week 2).
#
# "Stable Diffusion" = Latent Diffusion Model (LDM) + CLIP text conditioning
# + a particular VAE and U-Net checkpoint (SD 1.5, SDXL, etc.).
#
#
# ---------------------------------------------------------------------------
# 3) TRAINING OBJECTIVES — what gets trained, and what stays frozen
# ---------------------------------------------------------------------------
#
# Stage 1 — VAE (done before Stable Diffusion; not in this notebook's loop)
#   Loss ≈ "reconstruct the image" + KL penalty (same spirit as VAE notebook)
#   Output: encoder/decoder that speak "latent language."
#
# Stage 2 — Latent diffusion U-Net (the main SD training)
#   1. Take image x, encode: z = Enc(x)
#   2. Pick random timestep t, add noise: z_t = √ᾱ z + √(1−ᾱ) ε
#   3. Encode prompt (optional caption from dataset): c = TextEnc("a photo of …")
#   4. U-Net predicts ε̂ = UNet(z_t, t, c)
#   5. Loss = MSE(ε, ε̂)   ← same homework as DDPM, plus text conditioning
#
# Text conditioning trick — classifier-free guidance (CFG) at train time:
#   Sometimes drop the text (replace with empty prompt) so the net also
#   learns an "unconditional" denoiser. At inference we blend both:
#
#     ε̂ = ε̂_uncond + w · (ε̂_cond − ε̂_uncond)
#
#   w = guidance scale (often 7–12). Higher w = cling harder to the prompt,
#   but too high → oversaturated, weird artifacts.
#
# What we do NOT train in a typical SD fine-tune:
#   VAE and CLIP stay frozen. Only U-Net (or LoRA adapters on it) updates.
#
#
# ---------------------------------------------------------------------------
# 4) INFERENCE (SAMPLING) — text → image in practice
# ---------------------------------------------------------------------------
#
#   prompt ──▶ tokenize ──▶ CLIP ──▶ text embeddings c
#
#   z_T ~ N(0, I)     random latent snow, shape [1, 4, 64, 64] for 512² output
#
#   for t = T, T−1, …, 1:                    (often T=50 with DDIM, not 1000)
#       ε̂ = UNet(z_t, t, c)                  with CFG if w > 1
#       z_{t−1} = Scheduler.step(ε̂, z_t, t)
#
#   image = VAE.decode(z_0)
#
# Compare to notebook 18:
#
#   DDPM MNIST          Stable Diffusion
#   ─────────────────   ─────────────────────────────
#   pixels 32×32×1      latent 64×64×4  → decode to 512×512×3
#   no text             prompt via CLIP + cross-attention
#   train U-Net         use pretrained U-Net (+ optional LoRA)
#   ~1000 steps         ~20–50 steps with fast scheduler
#
# This notebook: we LOAD a pretrained pipeline (Hugging Face diffusers),
# not train from scratch. Goal = see the pieces work and understand the flow.
#
#
# ---------------------------------------------------------------------------
# One table
# ---------------------------------------------------------------------------
#   Pixel DDPM     denoise full-resolution RGB directly
#   Latent DDPM    denoise VAE latent, then decode
#   VAE            zip/unzip between pixel ↔ latent (frozen)
#   CLIP           prompt → vectors (frozen)
#   U-Net          predict noise in latent space, listen to text
#   Scheduler      math to go z_t → z_{t−1} (not learned)
#   CFG            nudge generation toward the prompt at sample time
#   Sample         snow latent + prompt + peel + decode → image
#
# Next cell: install diffusers, load StableDiffusionPipeline, generate from a prompt.


In [ ]:
# =============================================================================
# SETUP — load a pretrained Stable Diffusion pipeline (no training here)
# =============================================================================
#
# diffusers = Hugging Face library that bundles VAE + U-Net + CLIP + scheduler
# into one StableDiffusionPipeline object.
#
# First run downloads ~4–5 GB (SD 1.5). Needs disk space + patience.
# CPU works for tiny demos but is VERY slow; GPU/MPS is strongly preferred.
#
# Do NOT upgrade/reload NumPy inside a running kernel — that breaks C
# extensions (e.g. numpy.linalg._umath_linalg.lstsq). Fix versions in a
# terminal, then Kernel → Restart.
#

import sys
import numpy as np

print(f"Python: {sys.executable}")
print(f"NumPy:  {np.__version__}")

# Soft check only — never pip-install + reimport mid-session.
if int(np.__version__.split(".")[0]) < 2:
    raise RuntimeError(
        "This notebook expects NumPy >= 2.0 in the project .venv.\n"
        "In a terminal:\n"
        '  .venv/bin/pip install -U "numpy>=2.0,<3" "pandas>=2.2"\n'
        "Then: Kernel → Restart, and re-run this cell.\n"
        "(Do not upgrade NumPy from inside the notebook.)"
    )

import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
# Apple Silicon: try device = "mps" if torch.backends.mps.is_available()

print(f"Using device: {device}")
print("StableDiffusionPipeline import OK")
print("Next: load pipeline, pass a prompt, call pipe(...).")

torch.manual_seed(42)


In [ ]:
# Load the Pipeline
# We'll load the Stable Diffusion v1.5 model (or v2.1, but v1.5 is widely used). The pipeline downloads the model if not cached.

model_id = "runwayml/stable-diffusion-v1-5"

# Load pipeline with float16 for faster inference on GPU
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,  # disable for faster loading; can be enabled
)
pipe = pipe.to(device)

In [ ]:
# Generate an Image

prompt = "a photograph of an astronaut riding a horse on Mars"
image = pipe(prompt).images[0]
image

# Display the image:
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
# Explore Key Parameters

# We'll examine the effect of:
# Guidance scale (guidance_scale)
# Number of inference steps (num_inference_steps)
# Different samplers (we'll stick to default PNDM for now)
# Negative prompts (to avoid unwanted elements)

def generate(prompt, negative_prompt="", guidance_scale=7.5, num_steps=50, seed=42):
    generator = torch.Generator(device=device).manual_seed(seed)
    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=num_steps,
        generator=generator,
    ).images[0]
    return image

# Vary guidance scale
prompt = "a beautiful landscape with mountains and a lake at sunset"
scales = [1.0, 3.0, 7.5, 15.0]
fig, axes = plt.subplots(1, len(scales), figsize=(16, 4))
for ax, s in zip(axes, scales):
    img = generate(prompt, guidance_scale=s, num_steps=30)
    ax.imshow(img)
    ax.set_title(f"Guidance {s}")
    ax.axis("off")
plt.show()

# Vary number of steps:
steps = [5, 10, 20, 50]
fig, axes = plt.subplots(1, len(steps), figsize=(16, 4))
for ax, n in zip(axes, steps):
    img = generate(prompt, num_steps=n, guidance_scale=7.5)
    ax.imshow(img)
    ax.set_title(f"Steps {n}")
    ax.axis("off")
plt.show()

# Use negative prompts:
prompt = "a portrait of a woman, digital art"
negative_prompt = "blurry, low quality, distorted, ugly"
img_wo_neg = generate(prompt, negative_prompt="")
img_w_neg = generate(prompt, negative_prompt=negative_prompt)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_wo_neg)
axes[0].set_title("Without negative prompt")
axes[0].axis("off")
axes[1].imshow(img_w_neg)
axes[1].set_title("With negative prompt")
axes[1].axis("off")
plt.show()

In [ ]:
# Inspecting the Components